In [ ]:
import numpy as np, os
from datetime import datetime
import torch

from bokeh.palettes import Category10
import bokeh.plotting as bk
bk.output_notebook()

In [ ]:
torch.get_num_threads() 

In [ ]:
eurcoins = [file for file in os.listdir('../data/Kraken_OHLCVT/') if file.endswith('EUR_1.csv')]

In [ ]:
len(eurcoins)

In [ ]:
def read_historical_data(path):
    data = []

    with open(path, 'r') as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split(',')
        if len(parts) != 7 or parts[0] == 'time':
            continue
        timestamp = datetime.fromtimestamp(int(parts[0]))
        data.append({
            "time": timestamp,
            "open": float(parts[1]),
            "high": float(parts[2]),
            "low": float(parts[3]),
            "close": float(parts[4]),
            "volume": float(parts[5]),
            "trades": float(parts[6])
        })

    return data

In [ ]:
pairs = {
    "Bitcoin":   "XBTEUR",
    "Ethereum":  "ETHEUR",
    "Ripple":    "XRPEUR",
    "Cardano":   "ADAEUR",
    "Polkadot":  "DOTEUR",
    "Chainlink": "LINKEUR",
    "Litecoin":  "LTCEUR",
    "Solana":    "SOLEUR",
    "Stellar":   "XLMEUR",
    "TRON":      "TRXEUR",
    "Monero":    "XMREUR",
    "Cosmos":    "ATOMEUR",
    "Dogecoin":  "DOGEEUR"
}

In [ ]:
data = {}

for name, symbol in pairs.items():
    try:
        print(f"Reading data for {name} ({symbol})...")
        path = f'../data/Kraken_OHLCVT/{symbol}_15.csv'
        d = read_historical_data(path, name)
        data[name] = sorted(d, key=lambda x: x['time'])
    except FileNotFoundError:
        print(f"Data file for {name} ({symbol}) not found.")

In [ ]:
coin = "Bitcoin"
skip = 1

t = [d['time']  for d in aligned_data[coin][::skip]]
c = [d['close'] for d in aligned_data[coin][::skip]]

In [ ]:
fig = bk.figure(x_axis_type='datetime', title=f'{coin} Close Prices', height=400, width=1000)
fig.scatter(t, c, size=1)
bk.show(fig)

In [ ]:
min_times = [data[name][0]['time'] for name in data]
min_time = max(min_times)
min_time

In [ ]:
max_times = [data[name][-1]['time'] for name in data]
max_time = min(max_times)

In [ ]:
aligned_data = {}

for name in data:
    print(f"{name}: {data[name][0]['time']} to {data[name][-1]['time']}")
    aligned_data[name] = [d for d in data[name] if d['time'] >= min_time and d['time'] <= max_time]

In [ ]:
from datetime import timedelta

In [ ]:
min_time

In [ ]:
aligned_data["Bitcoin"][3169]

In [ ]:
def align_data(data, interval):
    aligned_data, dt = {}, timedelta(minutes=interval)

    min_time = max([data[name][ 0]['time'] for name in data])
    max_time = min([data[name][-1]['time'] for name in data])

    for name in data:
        print(f"Aligning data for {name}...")
        print(f"Total range for {name}: {data[name][0]['time']} to {data[name][-1]['time']}")
        filtered_data = [d for d in data[name] if d['time'] >= min_time and d['time'] <= max_time]

        new_data = []
        i, t = 0, min_time
        while t < max_time:
            if filtered_data[i]['time'] == t:
                i += 1
            new_data.append({**filtered_data[i].copy(), 'time': t})
            t += dt
        
        aligned_data[name] = new_data

In [ ]:
align_data(data, interval=15)

In [ ]:
import torch
PI = 3.141592653589793238462

In [ ]:
def get_time_vector(time: datetime) -> torch.Tensor:
    """
    Generate a vector of datetime timestamps from start to end at given minute intervals.

    Parameters
    ----------
    time : datetime
        A datetime object.

    Returns
    -------
    torch.Tensor
        1D tensor of datetime timestamps.
    """
    dateiso = time.isocalendar()

    total_weeks = datetime(time.year, 12, 31).isocalendar().week

    period_day = (time.hour + (time.minute + time.microsecond / 1e6) / 60 / 60) / 24
    period_week = (dateiso.weekday - 1 + period_day) / 7
    period_year = (dateiso.week - 1 + period_week) / total_weeks

    angles = torch.tensor([period_day, period_week, period_year], dtype=torch.float32)
    
    return torch.stack([torch.sin(2 * PI * angles), torch.cos(2 * PI * angles)]).T

In [ ]:
time_ = get_time_vector(datetime(2023, 12, 31, 6, 0))[None]
time_.shape

In [ ]:
figs = []

s = torch.linspace(0, 2 * PI, 100)
for i in range(3):
    fig = bk.figure(title=f'Time Vector Component {i}', height=300, width=320, x_range=(-1.1,1.1), y_range=(-1.1,1.1))
    fig.line(torch.cos(s), torch.sin(s), line_width=1, color="gray")
    fig.scatter(time_[:,i,0], time_[:,i,1], size=5)
    figs.append(fig)

bk.show(bk.row(figs))